In [1]:
import sys
sys.path.insert(0, '../../gofher')

import os
import matplotlib.image as mpimg
from astropy.visualization import make_lupton_rgb

from gofher import run_gofher, run_gofher_with_parameters
from visualize import visualize
from file_helper import write_csv,check_if_folder_exists_and_create, construct_csv_dict
from spin_parity import read_spin_parity_galaxies_label_from_csv, standardize_galaxy_name
from sparcfire import read_sparcfire_galaxy_csv, get_ref_band_and_gofher_params
from matrix import create_centered_mesh_grid, create_minor_axis_angle_matrix, create_dist_matrix

from fits import write_fits

In [2]:
survery_to_use = "sdss" #Note: for sdss we are not using u do to poor quality

BANDS_IN_ORDER = ['g','r','i','z'] #Important: Must stay in order of BLUEST to REDDEST Waveband (Editting this will cause gofher to no longer correctly evaluate redder side of galaxy)
REF_BANDS_IN_ORDER = ['r','i','z','g'] #The prefernce each waveband being choosen as refernce band from highest priority to lowest priority

In [3]:
#figures_to_run_on = ["table2","table3","table4","table5"]
figures_to_run_on = ["figure9"]

In [4]:
#panstarrs:
bin_size = None #None or a positive integer

#sdss:
#NOTE: sdss needs to flip color image
##bin_size = 4 #should be 4
#Source: "The median seeing of all SDSS imaging data (using the psfWidth metric) is 1.32 arcseconds in the r-band."
#"The pixel size in the Sloan Digital Sky Survey (SDSS) is 0.396 arcseconds per pixel" - https://classic.sdss.org/dr3/instruments/imager/
bin_prior_to_param_fitting = True

In [5]:
generate_verbose_csv = True
generate_ebm_csv = False
generate_params_csv = True
generate_visualization = True
save_visualization = True
generate_gamma_csv = True

In [6]:
#Important: Make sure you update these values:
blur_sdss_fits_folder = "E:\\grad_school\\research\\spin_parity_blurring\\sdss_output"
blur_sdss_folder = "E:\\grad_school\\research\\spin_parity_blurring\\sparcfire_sdss_output"
blur_sparcfire_folder = "E:\\grad_school\\research\\spin_parity_blurring\\sparcfire_sdss_output"
path_to_output = "C:\\Users\\school\\Desktop\\github\\gofher-data\\sdss\\blur_folder_9"
gofher_labels_folder = "C:\\Users\\school\\Desktop\\github\\gofher-data\\sdss\\sparcfire_0_25"
path_to_catalog_data = "..\\..\\..\\spin-parity-catalog-data"
fits_save_path = "E:\\grad_school\\research\\winter_2026\\blur_fits_save_path\\"

In [7]:
def get_blur_folder(the_sn,the_psf):
    return "psf_{}_background_{}".format(str(the_psf),str(the_sn))

def get_sparcfire_galaxy_csv_path(table_name,the_sn,the_psf):
    return os.path.join(blur_sdss_folder,get_blur_folder(the_sn,the_psf),table_name,"G.out","galaxy.csv")

In [8]:
def construct_image(gal):
    for band in ["g","r","i"]:
        if band not in gal.bands:
            the_keys = list(gal.bands.keys())
            return gal[the_keys[0]].data

    g = gal.bands["g"].data
    r = gal.bands["r"].data*0.8
    i = gal.bands["i"].data*0.7

    return make_lupton_rgb(i, r, g, Q=10, stretch=0.3, minimum=0.0)

In [9]:

folder_map = {"table2":"figure8",
              "table3":"figure9",
              "table4":"figure10",
              "table5":"figure11",
              "figure9":"table3"}

def get_path_to_catalog_csv(figure_to_run_on):
    return os.path.join(path_to_catalog_data,"catalog","{}.csv".format(figure_to_run_on))

def get_paper_dark_side_labels(figure_to_run_on):
    #temp procedure for blending two:
    if figure_to_run_on == "table3":
        figure_to_run_on = "figure9"
    return read_spin_parity_galaxies_label_from_csv(get_path_to_catalog_csv(figure_to_run_on))

def get_fits_path(name,band,blur_folder,figure_to_run_on):
    """the file path of where existing fits files can be found"""
    return os.path.join(blur_sdss_fits_folder,blur_folder,figure_to_run_on,name,"{}_{}.fits".format(name,band))

def get_galaxy_list(blur_folder,figure_to_run_on):
    the_directory = os.path.join(blur_sdss_fits_folder,blur_folder,figure_to_run_on)
    return os.listdir(the_directory)

def get_sparcfire_path(blur_folder,figure_to_run_on): #temp
    return os.path.join(blur_sparcfire_folder,blur_folder,figure_to_run_on,"G.out","galaxy.csv")

def get_path_to_output(blur_folder, figure_to_run_on):
    if figure_to_run_on in folder_map:
        figure_to_run_on = folder_map[figure_to_run_on]
    return os.path.join(path_to_output,blur_folder)

def get_fits_save_path(blur_folder,figure_to_run_on,name):
    if figure_to_run_on in folder_map:
        figure_to_run_on = folder_map[figure_to_run_on]
    return os.path.join(fits_save_path,blur_folder,figure_to_run_on,name)


def get_gofher_labels(figure_to_run_on):
    if figure_to_run_on in folder_map:
        figure_to_run_on = folder_map[figure_to_run_on]
    path = os.path.join(gofher_labels_folder,f"{figure_to_run_on}_verbose.csv")
    return construct_csv_dict(path,"name","GOFHER_label")

In [10]:
def get_visulization_save_path_folder(name,figure_to_run_on, blur_folder):
    if figure_to_run_on in folder_map:
        figure_to_run_on = folder_map[figure_to_run_on]
    return os.path.join(path_to_output,blur_folder,figure_to_run_on)

def _ensure_path_exists(path_to_output,make_ouput_folder_if_not_exists=True):
    if not make_ouput_folder_if_not_exists:
        raise ValueError("The path output is not found {} - make sure you update path_to_output".format(path_to_output))
    
    if not os.path.exists(path_to_output):
        os.makedirs(path_to_output)

    

In [11]:
def run_gofher_on_catalog(figure_to_run_on,blur_folder,bulge_disk_f=1.0):
    paper_labels = get_paper_dark_side_labels(folder_map[figure_to_run_on])
    #gofher_labels = get_gofher_labels(figure_to_run_on)
    sparcfire_gals = read_sparcfire_galaxy_csv(get_sparcfire_path(blur_folder,figure_to_run_on))

    verbose_header = []
    verbose_rows = []

    params_header = []
    params_rows = []

    gamma_header = []
    gamma_rows = []

    i = 1

    path_to_output = get_path_to_output(blur_folder, figure_to_run_on)
    #print(path_to_output)
    #return
    _ensure_path_exists(path_to_output)

    def get_fits_path_cat(name,band):
        """the file path of where existing fits files can be found"""
        return get_fits_path(name,band,blur_folder,figure_to_run_on)

    galaxies = get_galaxy_list(blur_folder,figure_to_run_on)
    for name in galaxies:

        if standardize_galaxy_name(name) not in paper_labels:
            print("skipping",name)
            continue

        print(name, i,"of",len(galaxies))

        try:
            fits_path = get_fits_save_path(blur_folder,figure_to_run_on,name)
            _ensure_path_exists(fits_path)
            

            paper_label = paper_labels[standardize_galaxy_name(name)]

            ref_band, inital_gofher_params = get_ref_band_and_gofher_params(sparcfire_gals[name],REF_BANDS_IN_ORDER,bulge_disk_f)
            gal = run_gofher_with_parameters(name,get_fits_path_cat,BANDS_IN_ORDER,ref_band,inital_gofher_params,paper_label=paper_label)

            import matplotlib.pyplot as plt 

            el_mask = gal.create_ellipse()
            pos_mask, neg_mask = gal.create_bisection()


            write_fits(os.path.join(fits_path,"el_mask.fits"),el_mask.astype(float))
            write_fits(os.path.join(fits_path,"pos_mask.fits"),pos_mask.astype(float))
            write_fits(os.path.join(fits_path,"neg_mask.fits"),neg_mask.astype(float))

            h = gal.gofher_params.x
            k = gal.gofher_params.y
            theta = gal.gofher_params.theta
            shape = gal.bands[gal.ref_band].get_shape()
            xv, yv = create_centered_mesh_grid(h,k,shape)

            dist = create_dist_matrix(xv,yv)
            ang = create_minor_axis_angle_matrix(h,k,theta,shape)

            write_fits(os.path.join(fits_path,"dist.fits"),dist.astype(float))
            write_fits(os.path.join(fits_path,"ang.fits"),ang.astype(float))
            
            for band_pair_key in gal.band_pairs:
                the_diff = gal.band_pairs[band_pair_key].diff_image
                write_fits(os.path.join(fits_path,f"{band_pair_key}.fits"),the_diff)

        except Exception as e:
            print(e)
        i += 1

In [12]:
#if not os.path.exists(path_to_catalog_data):
#    raise ValueError("The path to the catalog is not found {} - make sure you update path_to_catalog_data".format(path_to_catalog_data))

sn_list = [0.5, 0.25,0.125,0.0625, 1, 2, 4, 8, 16, 32, 64, 128, 256]
psf_list = [4.0, 5.6, 8.0, 11.3,16.0, 22.6, 32.0, 45.2, 64.0, 90.5, 128.0]

for sn in sn_list:
    for psf in psf_list:
        for figure_to_run_on in figures_to_run_on:
            blur_folder = get_blur_folder(sn,psf)
            run_gofher_on_catalog(figure_to_run_on,blur_folder,bulge_disk_f=0.25)

IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_0.5\figure9\NGC3344\NGC3344_z.fits does not exist
NGC3346 4 of 14
NGC3351 5 of 14
NGC3359 6 of 14
zero-size array to reduction operation minimum which has no identity
NGC3367 7 of 14
NGC3381 8 of 14
NGC3395 9 of 14
'NGC3395'
NGC3423 10 of 14
NGC3445 11 of 14
PGC39728 12 of 14
PGC46767 13 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_0.5\figure9\PGC46767\PGC46767_g.fits does not exist
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_0.5\figure9\PGC46767\PGC46767_z.fits does not exist
PGC49906 14 of 14
IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_5.6_background_0.5\figure9\NGC3344\NGC3344_z.fits does not